<a href="https://colab.research.google.com/github/umetaisa2004-commits/content-ops-starter/blob/main/20261019_Photron%E4%BB%AE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 「report-def-」で始まるファイル（ZIP含む）とフォルダをすべて削除
!rm -rf report-def-*

# 特定のZIPファイルとフォルダをまとめて削除する場合の例
!rm -rf *.zip target_folder_name

In [ ]:
import cv2
import numpy as np
import glob
import os
import pandas as pd
import re
try:
    from google.colab import files
except ImportError:
    pass

# --- 1. 処理設定 ---
folder_path = "/content"

# /content 直下の bmpファイルをすべて取得
bmp_paths = glob.glob(os.path.join(folder_path, "*.bmp"))

scale_factor = 38.09  # 1px = 38.09µm

def get_sec(filepath):
    # ファイル名（例: 50.bmp）から数字だけを抽出
    match = re.search(r'\d+', os.path.basename(filepath))
    return int(match.group()) if match else 999999

# 秒数の小さい順（50, 100, 150...）にソート
bmp_paths = sorted(bmp_paths, key=get_sec)

print(f"全 {len(bmp_paths)} 枚のbmpファイルを処理します...\n")

# --- 2. 各bmpファイルの解析 ---
all_results = []

for bmp_path in bmp_paths:
    filename = os.path.basename(bmp_path)
    sec_val = get_sec(bmp_path)

    img_gray = cv2.imread(bmp_path, cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        print(f"⚠️ 読み込めませんでした: {filename}")
        continue

    # 適応的二値化
    blurred = cv2.GaussianBlur(img_gray, (3, 3), 0)
    flocs_mask = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 5)

    contours, _ = cv2.findContours(flocs_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    diameters_um = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 0:
            diameter_px = 2 * np.sqrt(area / np.pi)
            diameter_um = diameter_px * scale_factor

            # 10mm (10,000µm) 以上のノイズ・巨大な塊は除外
            if diameter_um < 10000:
                diameters_um.append(diameter_um)

    count = len(diameters_um)

    if count > 0:
        d_array = np.array(diameters_um)
        d10, d30, d50, d70, d90 = np.percentile(d_array, [10, 30, 50, 70, 90])
        mean_d = np.mean(d_array)
        max_d = np.max(d_array)
        min_d = np.min(d_array)
        span = (d90 - d10) / d50 if d50 > 0 else np.nan
    else:
        d10 = d30 = d50 = d70 = d90 = mean_d = max_d = min_d = span = np.nan

    all_results.append({
        "ファイル名": filename,
        "秒数 (s)": sec_val,
        "有効粒子数": count,
        "D10 (µm)": d10,
        "D30 (µm)": d30,
        "D50 (µm)": d50,
        "D70 (µm)": d70,
        "D90 (µm)": d90,
        "平均径 (µm)": mean_d,
        "最大径 (µm)": max_d,
        "最小径 (µm)": min_d,
        "Span (Polydispersity)": span
    })

# --- 3. 結果の出力とダウンロード ---
if len(all_results) > 0:
    df_all = pd.DataFrame(all_results)

    excel_filename = "Floc_Analysis_Result_DirectBMP.xlsx"
    df_all.to_excel(excel_filename, sheet_name="解析結果", index=False)

    print(f"\n✅ 解析完了。Excelファイルを保存しました: {excel_filename}")

    # Colab環境なら自動ダウンロード
    if 'google.colab' in str(get_ipython()):
        files.download(excel_filename)
else:
    print("\n❌ bmpファイルが見つからなかったか、解析できませんでした。")

全 18 枚のbmpファイルを処理します...


✅ 解析完了。Excelファイルを保存しました: Floc_Analysis_Result_DirectBMP.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>